In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.preprocessing import StandardScaler

from collections import Counter

import pymc as pm

import arviz as az
import math
import csv

independent=0

param_rng = np.random.default_rng(12345)
np.random.seed(123456)
print(param_rng)

from scipy.stats import multivariate_normal, invwishart, wishart, dirichlet, rv_discrete, beta, gamma

import sys
print(sys.version)

In [ ]:
healthdata = pd.read_csv("NHANES_adults_data_preprocessed.csv")

healthdata=healthdata.drop(columns=['Unnamed: 0','id_study','samplewt_anthro', 'samplewt_hba1c','samplewt_bp','samplewt_chol'])

healthdata=healthdata.drop(columns=['crp'])

print('done')

In [ ]:
print(healthdata.describe())

In [ ]:
def conditionbmi(s):
    if (s['bmi'] < 18.5):
        return 0
    elif (s['bmi'] >= 18.5) and (s['bmi'] < 25) :
        return 1
    elif (s['bmi'] >= 25) and (s['bmi'] < 30) : 
        return 2
    else:
        return 3
    
    
def conditionwhtr(s):
    if (s['WHtR'] < 0.5):
        return 0
    else:
        return 1
    
def conditionsbpdbp(s):
    if (s['sbp']<130 and s['dbp']<80):
        return 0
    elif ((s['sbp']>=130 and s['sbp']<140) and (s['dbp']<90)):
        return 1
    elif ((s['sbp']<140) and (s['dbp']>=80 and s['dbp']<90)):
        return 1
    elif ((s['sbp']>=140 and s['sbp']<160) and (s['dbp']<100)):
        return 2
    elif ((s['sbp']<160) and (s['dbp']>=90 and s['dbp']<100)):
        return 2
    elif ((s['sbp']>=160) or (s['dbp']>=100)):
        return 3
    else:
        return 3
    
def conditionhdl(s):
    if (s['hdl']<1.293):
        return 0
    else:
        return 1
    
def conditionnon_hdl(s): 
    if (s['non_hdl']<3.36):
        return 0
    elif (s['non_hdl']>=3.36 and s['non_hdl']<5.69):
        return 1
    elif (s['non_hdl']>=5.69):
        return 2
    else:
        return 3
    
def conditionhba1c(s):
    if (s['hba1c']<6.5):
        return 0
    elif (s['hba1c']>=6.5 and s['hba1c']<10.0):
        return 1
    elif (s['hba1c']>=10.0):
        return 2
    else:
        return 3
    
    
def conditioneGFR(s): 
    if (s['eGFR']<60):
        return 0
    elif (s['eGFR']>=60 and s['eGFR']<90):
        return 1
    elif (s['eGFR']>=90):
        return 2
    else:
        return 3
    
def conditionage(s):
    if (s['age'] < 40):
        return 0
    elif (s['age'] >= 40) and (s['age'] < 60) :
        return 1
    else:
        return 2
    
def combinesmoking(s):
    if (s['smoker']==0 and s['smoke_ever']==0):
        return 0 
    elif (s['smoker']==0 and s['smoke_ever']==1):
        return 1 
    elif (s['smoker']==1 and s['smoke_ever']==1):
        return 2 
    else:
        return 3 
    
    
healthdata['bmicat'] = healthdata.apply(conditionbmi, axis=1)

healthdata['whtrcat'] = healthdata.apply(conditionwhtr, axis=1)

healthdata['bpcat'] = healthdata.apply(conditionsbpdbp, axis=1)

healthdata['hdlcat'] = healthdata.apply(conditionhdl, axis=1)

healthdata['non_hdlcat'] = healthdata.apply(conditionnon_hdl, axis=1)

healthdata['hba1ccat'] = healthdata.apply(conditionhba1c, axis=1)

healthdata['eGFRcat'] = healthdata.apply(conditioneGFR, axis=1)

healthdata['agecat'] = healthdata.apply(conditionage, axis=1)

healthdata['smokecombined']=healthdata.apply(combinesmoking, axis=1)

print('done')

In [ ]:
print(healthdata.shape[0])
healthdata=healthdata.dropna()
print(healthdata.shape[0])

healthdata.describe()

healthdata=healthdata[healthdata['sex']==1]

print(healthdata.shape[0]) 

healthdata=healthdata[healthdata['agecat']==1]

print(healthdata.shape[0]) 

In [ ]:
print(Counter(healthdata['smokecombined']))

print(Counter(healthdata['smokecombined']))

print(Counter(healthdata['is_black']))

print(Counter(healthdata['self_diab']))

print(Counter(healthdata['self_hyper']))

print(Counter(healthdata['drug_hyper']))

print(Counter(healthdata['drug_chol']))

print(Counter(healthdata['drug_diab_insu']))

print(Counter(healthdata['drug_diab_pill']))

print(Counter(healthdata['ethnicity']))

print(Counter(healthdata['education']))

print(Counter(healthdata['self_mi']))

print(Counter(healthdata['self_stroke']))

print(Counter(healthdata['self_heart_failure']))

In [ ]:
discrete_cols=['smoker', 'smoke_ever', 'is_black', 'self_diab', 'self_hyper', 'drug_hyper', 'drug_chol', 'drug_diab_insu', 'drug_diab_pill', 'self_mi', 'self_stroke', 'self_heart_failure', 'bmicat', 'whtrcat', 'bpcat', 'hdlcat', 'non_hdlcat', 'hba1ccat', 'eGFRcat', 'agecat', 'smokecombined']

for col in discrete_cols:
    healthdata[col] = healthdata[col].astype(int)

In [ ]:
if independent==1:
    allcategoricalvalues=[0]  
else:
    allcategoricalvalues=[0,1]

modelled_continuous_columns=['bmi','height','WHtR','hba1c','hdl','non_hdl','sbp','dbp','eGFR','pulse']
modelled_categorical_columns=['smokecombined']

def merge_columns(row):
    return '_'.join(row.astype(str))

healthdata['combinedcategories'] = healthdata[modelled_categorical_columns].apply(merge_columns, axis=1)


modelled_categorical_columns_combined=['combinedcategories']

print(healthdata['combinedcategories'] )

print(Counter(healthdata['combinedcategories']))

In [ ]:
fulldataframe=pd.DataFrame(healthdata, columns = modelled_continuous_columns+modelled_categorical_columns+modelled_categorical_columns_combined)

N=healthdata.shape[0]
print(fulldataframe)

In [ ]:
scaler = StandardScaler()

fulldataframe[modelled_continuous_columns] = scaler.fit_transform(fulldataframe[modelled_continuous_columns])

print('done standardising data')


fulldataframe.describe()

combinedcategories=np.array(fulldataframe['combinedcategories'])

continuousdata=np.array(fulldataframe[modelled_continuous_columns])

print(np.shape(combinedcategories))

print(np.shape(continuousdata))

print(np.mean(continuousdata, axis=0))
print(np.std(continuousdata,axis=0))

print(np.unique(combinedcategories,return_counts=True))

allcatvals=np.unique(combinedcategories,return_counts=True)

print('all combined categorical values')
print(allcatvals[0])

(combinedcategories==allcatvals[0][1])

print(len(allcatvals[0]))

print(np.unique(combinedcategories==allcatvals[0][1], return_counts=True))

In [ ]:
fulldataframegroupbymeans = fulldataframe.groupby(['combinedcategories'], as_index=False)[modelled_continuous_columns].mean()

fulldataframegroupbystd = fulldataframe.groupby(['combinedcategories'], as_index=False)[modelled_continuous_columns].std() 

print('data means grouped by categorical variable')
print(fulldataframegroupbymeans)
print(type(fulldataframegroupbymeans))

print('data standard deviations grouped by categorical variable')
print(fulldataframegroupbystd)

In [ ]:
categorical_columns_dims=[]

for catcols in modelled_categorical_columns:
    print(fulldataframe[catcols].unique())
    print(len(fulldataframe[catcols].unique()))
    categorical_columns_dims.append(len(fulldataframe[catcols].unique()))

print('check categorical columns')
print(categorical_columns_dims)
print(len(categorical_columns_dims))

print(tuple([10001,10]+categorical_columns_dims+[continuousdata.shape[1]])) 

In [ ]:
Psiindependentofk=0

catvaluetoindex = dict(zip(allcatvals[0], list(range(0, len(allcatvals[0])))))

mudependentong=0
Sigmadependentong=0
pidependentong=0

if mudependentong==0:
    catvaluetoindexmu = dict(zip(allcatvals[0], [0] * len(allcatvals[0]) ))
    allcatvalsmu=np.array(['0'])

    catvaluetoindexmu['0']=0
    combinedcategoriesmu=np.array(['0']*N)
else:
    allcatvalsmu=allcatvals
    catvaluetoindexmu = dict(zip(allcatvals[0], list(range(0, len(allcatvals[0])))))
    combinedcategoriesmu=combinedcategories
    
if Sigmadependentong==0:
    catvaluetoindexSigma = dict(zip(allcatvals[0], [0] * len(allcatvals[0]) ))
    allcatvalsSigma=np.array(['0'])
    catvaluetoindexSigma['0']=0
    combinedcategoriesSigma=np.array(['0']*N)
else:
    allcatvalsSigma=allcatvals
    catvaluetoindexSigma = dict(zip(allcatvals[0], list(range(0, len(allcatvals[0])))))
    combinedcategoriesSigma=combinedcategories
    
if pidependentong==0:
    catvaluetoindexpi = dict(zip(allcatvals[0], [0] * len(allcatvals[0]) ))
    allcatvalspi=np.array(['0'])
    catvaluetoindexpi['0']=0
    combinedcategoriespi=np.array(['0']*N)
else:
    allcatvalspi=allcatvals
    catvaluetoindexpi = dict(zip(allcatvals[0], list(range(0, len(allcatvals[0])))))
    combinedcategoriespi=combinedcategories

In [ ]:
#initialise variational parameters to be updated

K = 10 
p = len(modelled_categorical_columns)
q = len(modelled_continuous_columns)

iterations = 3000 

mdims = tuple([iterations,K] + [len(allcatvalsmu[0])] + [q]) 
betadims=tuple([iterations,K] + [len(allcatvalsmu[0])]) 

m_kg_hat=np.zeros(mdims)
beta_kg_hat=np.zeros(betadims)

Phidims=tuple([iterations, K, len(allcatvalsSigma[0]), q, q])

nudims=tuple([iterations, K, len(allcatvalsSigma[0])])

Phi_k_hat=np.zeros(Phidims) 
nu_k_hat=np.zeros(nudims)

piparamdims=tuple([iterations]+ [K, len(allcatvalspi[0])]) 

piparam=np.zeros(piparamdims)

if Psiindependentofk==1:
    Psiparamdims=tuple([iterations, 1, p] + [np.max(categorical_columns_dims)]) #subgroup removed
else:
    Psiparamdims=tuple([iterations, K, p] + [np.max(categorical_columns_dims)]) #subgroup removed

Psiparam=np.zeros(Psiparamdims)

zparamdims=tuple([N, K])

r_ik=np.zeros(zparamdims)+0.1
    
rho_ik=np.zeros(zparamdims)
logrho_ik=np.zeros(zparamdims)

from kmodes.kprototypes import KPrototypes

cat_indices = [fulldataframe[modelled_continuous_columns+modelled_categorical_columns].columns.get_loc(col) for col in modelled_categorical_columns] 

kproto = KPrototypes(n_clusters=K, init='Cao', n_jobs=-1, random_state=42)
kprotoclusters = kproto.fit_predict(fulldataframe[modelled_continuous_columns+modelled_categorical_columns].to_numpy(), categorical=cat_indices)


for i in range(0,N):
    r_ik[i,kprotoclusters[i]]=0.9

print('done')

In [ ]:
mu_kg_prior=np.zeros([K] + [len(allcatvalsmu[0])] + [q]) 

print('check mu_kg_prior')

m_kg_perturbation=np.random.multivariate_normal(mean=np.zeros(q),cov=0.25*np.eye(q),size=K)

if mudependentong==0:
    for k in range(0,K):
        for g in allcatvalsmu[0]:
            mu_kg_prior[k,catvaluetoindexmu[g],:] = np.zeros(q) 
            print('k='+str(k)+' g='+g)
            print(mu_kg_prior[k,catvaluetoindexmu[g],:])

else:
    for k in range(0,K):
        for g in allcatvalsmu[0]:
            mu_kg_prior[k,catvaluetoindexmu[g],:] = np.zeros(q) 
            print('k='+str(k)+' g='+g)
            print(mu_kg_prior[k,catvaluetoindexmu[g],:])



beta_kg_prior=1.0

Phi=0.25*np.eye(q) 
nu=q + K + 1

piparamprior=1/K

Psiparamprior=np.zeros(tuple([K, p] + [np.max(categorical_columns_dims)])) 

for k in range(0,K):
    for j in range(0,p):
        for g_j in range(0,categorical_columns_dims[j]):
            Psiparamprior[k,j,g_j]=1/categorical_columns_dims[j] 

print('check Psiparamprior')
print(Psiparamprior)
for k in range(0,K): 
    for j in range(0,p):
        print('k='+str(k)+' j='+str(j))
        print(Psiparamprior[k,j,:])

print('done')

In [ ]:
#VI updates
if continuousdata.shape[0] != N or fulldataframe.shape[0] != N or r_ik.shape[0] != N: #notebook
    raise ValueError(f'N={N} must equal the number of rows of the data used ({continuousdata.shape[0]} in continuousdata, {fulldataframe.shape[0]} in fulldataframe, {r_ik.shape[0]} in r_ik)') #notebook
from scipy.special import digamma
from scipy.special import multigammaln, gammaln
from scipy.special import logsumexp
from scipy.special import xlogy
from scipy.special import softmax
import pickle
import time

ELBOoldcontonly=0

ELBO=1+np.zeros(iterations)
ELBOold=-10000
improvement=100
iteration=0

Elnpi_k=np.zeros((K,len(allcatvalspi[0])))
Elnpsi_kjg=np.zeros(tuple([K, p] + [np.max(categorical_columns_dims)]))
ElndetLambda_k=np.zeros((K,len(allcatvalsSigma[0])))
ElndetLambda_k_digammaterm=np.zeros((K,len(allcatvalsSigma[0])))

def quadform(vector,matrix):
    return np.transpose(vector) @ matrix @ vector


start_time = time.time()
vi_compute_elapsed = 0.0
timerecord = np.zeros(iterations) 

locations_pi_g = {g: np.where(combinedcategoriespi == g)[0] for g in allcatvalspi[0]}
locations_mu_g = {g: np.where(combinedcategoriesmu == g)[0] for g in allcatvalsmu[0]}
locations_Sigma_g = {g: np.where(combinedcategoriesSigma == g)[0] for g in allcatvalsSigma[0]}
locations_cat_jg = {}
counts_cat_jg = {}
for j in range(0,p):
    for g_j in range(0,categorical_columns_dims[j]):
        locations_cat_jg[(j,g_j)] = np.where(fulldataframe[modelled_categorical_columns[j]] == g_j)[0]
        counts_cat_jg[(j,g_j)] = np.sum(fulldataframe[modelled_categorical_columns[j]] == g_j)

sigma_indices = np.array([catvaluetoindexSigma[c] for c in combinedcategoriesSigma])
mu_indices = np.array([catvaluetoindexmu[c] for c in combinedcategoriesmu])
pi_indices = np.array([catvaluetoindexpi[c] for c in combinedcategoriespi])
unique_sigma_indices, sigma_inverse_map = np.unique(sigma_indices, return_inverse=True)
unique_mu_indices, mu_inverse_map = np.unique(mu_indices, return_inverse=True)

categoricaldataframe = {
col: fulldataframe[col].to_numpy()
for col in modelled_categorical_columns
}

log_inv_det_Phi_prior = np.log(1/np.linalg.det(Phi))

dummyiteration=0
while iteration<iterations and (improvement>0.0001 or improvement<0): 

    print('dummy iteration')
    print(dummyiteration)
    


    iter_t0 = time.perf_counter()
    
    sum_r_pi_gk = {g: np.array([np.sum(r_ik[locations_pi_g[g],k]) for k in range(0,K)]) for g in allcatvalspi[0]}
    totsum_r_pi_g = {g: np.sum(r_ik[locations_pi_g[g],:], axis=None) for g in allcatvalspi[0]}
    sum_r_mu_gk = {g: np.array([np.sum(r_ik[locations_mu_g[g],k]) for k in range(0,K)]) for g in allcatvalsmu[0]}
    sum_r_Sigma_gk = {g: np.array([np.sum(r_ik[locations_Sigma_g[g],k]) for k in range(0,K)]) for g in allcatvalsSigma[0]}
    sum_r_cat_jgk = {key: np.array([np.sum(r_ik[idx,k]) for k in range(0,K)]) for key, idx in locations_cat_jg.items()}
    colsum_r_k = np.array([np.sum(r_ik[:,k]) for k in range(0,K)])
    

    for k in range(0,K):
        for g in allcatvalspi[0]:
            piparam[iteration,k,catvaluetoindexpi[g]]=piparamprior+sum_r_pi_gk[g][k]

    for k in range(0,K):
        for g in allcatvalsmu[0]:
            locations_g_idx = locations_mu_g[g] 

            beta_kg_hat[iteration,k,catvaluetoindexmu[g]]=beta_kg_prior+sum_r_mu_gk[g][k] 

            m_kg_hat[iteration,k,catvaluetoindexmu[g],:]=(beta_kg_prior*mu_kg_prior[k,catvaluetoindexmu[g],:]+np.transpose(continuousdata[locations_g_idx,:]) @ r_ik[locations_g_idx,k])/(beta_kg_hat[iteration,k,catvaluetoindexmu[g]])
    
    Phi_update_mhatttermstore=np.zeros((K,q,q))
    Phi_update_mtermstore=np.zeros((K,q,q))
    Phi_update_xtermstore=np.zeros((K,q,q))
    

    for k in range(0,K):
        for g in allcatvalsSigma[0]:
            locations_g_idx = locations_Sigma_g[g] 
            
            nu_k_hat[iteration,k,catvaluetoindexSigma[g]]=nu+sum_r_Sigma_gk[g][k]
            
            m_kg_hat_vec=m_kg_hat[iteration,k,catvaluetoindexmu[g],:] 
            m_kg_hat_vec=np.reshape(m_kg_hat_vec,(m_kg_hat_vec.size,1))
            
            mu_kg_prior_vec=mu_kg_prior[k,catvaluetoindexmu[g],:]
            mu_kg_prior_vec=np.reshape(mu_kg_prior_vec,(mu_kg_prior_vec.size,1))
            
            Phi_update_mhattterm=(beta_kg_prior+sum_r_Sigma_gk[g][k])*m_kg_hat_vec @ np.transpose(m_kg_hat_vec) 
            
            Phi_update_mterm=beta_kg_prior*mu_kg_prior_vec @ np.transpose(mu_kg_prior_vec)
        
            X_g = continuousdata[locations_g_idx,:]
            r_g = r_ik[locations_g_idx,k]
            Phi_update_xterm = np.transpose(X_g) @ (r_g[:,np.newaxis]*X_g)
            
            Phi_k_hat[iteration,k,catvaluetoindexSigma[g],:]=Phi-Phi_update_mhattterm+Phi_update_mterm+Phi_update_xterm
            
        Phi_update_mhatttermstore[k,:]=Phi_update_mhattterm
        Phi_update_mtermstore[k,:]=Phi_update_mterm
        Phi_update_xtermstore[k,:]=Phi_update_xterm
    
    if Psiindependentofk==1:
        for j in range(0,p):
            for g_j in range(0,categorical_columns_dims[j]):
                Psiparam[iteration,0,j,g_j]=Psiparamprior[0,j,g_j] + counts_cat_jg[(j,g_j)]

                        
    else: 
        for k in range(0,K):
            for j in range(0,p):
                for g_j in range(0,categorical_columns_dims[j]):
                    Psiparam[iteration,k,j,g_j]=Psiparamprior[k,j,g_j]+sum_r_cat_jgk[(j,g_j)][k]
                    

    ElndetLambda_k_digammaterm=np.zeros((K,len(allcatvalsSigma[0])))
    for k in range(0,K):
        for g in allcatvalsSigma[0]:
            for i in range(1,q+1): 
                ElndetLambda_k_digammaterm[k,catvaluetoindexSigma[g]]=ElndetLambda_k_digammaterm[k,catvaluetoindexSigma[g]]+digamma((nu_k_hat[iteration,k,catvaluetoindexSigma[g]]+1-i)/2) #appears to be correct
    

    for k in range(0,K):
        for g in allcatvalspi[0]:
            Elnpi_k[k,catvaluetoindexpi[g]]=digamma(piparamprior+sum_r_pi_gk[g][k])-digamma(K*piparamprior+totsum_r_pi_g[g])
            
    
    log_inv_det_Phi_k_hat=np.zeros((K,len(allcatvalsSigma[0])))
    inv_Phi_k_hat_stored=np.zeros((K,len(allcatvalsSigma[0]),q,q))
    for k in range(0,K):
        for g in allcatvalsSigma[0]:
            log_inv_det_Phi_k_hat[k,catvaluetoindexSigma[g]]=np.log(1/np.linalg.det(Phi_k_hat[iteration,k,catvaluetoindexSigma[g],:]))
            inv_Phi_k_hat_stored[k,catvaluetoindexSigma[g],:]=np.linalg.inv(Phi_k_hat[iteration,k,catvaluetoindexSigma[g],:])
    
    for k in range(0,K):
        for g in allcatvalsSigma[0]:
            ElndetLambda_k[k,catvaluetoindexSigma[g]]=q*np.log(2)+log_inv_det_Phi_k_hat[k,catvaluetoindexSigma[g]]+ElndetLambda_k_digammaterm[k,catvaluetoindexSigma[g]] 
    
    if Psiindependentofk==1:
        for j in range(0,p):
            for g_j in range(0,categorical_columns_dims[j]):
                Elnpsi_kjg[0,j,g_j]=digamma(Psiparamprior[0,j,g_j]+counts_cat_jg[(j,g_j)])-digamma(np.sum(Psiparamprior[0,j,:])+N) 

    else:
        for k in range(0,K):
            for j in range(0,p):
                for g_j in range(0,categorical_columns_dims[j]):
                    Elnpsi_kjg[k,j,g_j]=digamma(Psiparamprior[k,j,g_j]+sum_r_cat_jgk[(j,g_j)][k])-digamma(np.sum(Psiparamprior[k,j,:])+colsum_r_k[k]) 

    


    ElnNx_all_ik = np.zeros((N, K))
   
    ELBOElnNx_i = 0
    for k in range(0, K):
    
        unique_inv_Phi = np.linalg.inv(Phi_k_hat[iteration, k, unique_sigma_indices, :, :])

        unique_nu = nu_k_hat[iteration, k, unique_sigma_indices] #even if this normally a 'scalr', unique_sigma_indices being [0] will cause this to also be [some scaler]
        unique_beta = beta_kg_hat[iteration, k, unique_mu_indices]
        unique_m = m_kg_hat[iteration, k, unique_mu_indices, :]
        unique_ElndetLambda = ElndetLambda_k[k, unique_sigma_indices]

        nu_values = unique_nu[sigma_inverse_map]
        beta_values = unique_beta[mu_inverse_map]
        m_vectors = unique_m[mu_inverse_map] 
        ElndetLambda_values = unique_ElndetLambda[sigma_inverse_map]

        
        term1 = -(q / 2) * np.log(2*np.pi)
        term2 = 0.5 * ElndetLambda_values
        
        diffs = continuousdata - m_vectors
        quad_base = np.empty(diffs.shape[0])                                  
        for Phi_index in range(unique_inv_Phi.shape[0]):                              
            sub_Phi = (sigma_inverse_map == Phi_index)                                   
            diffs_sub = diffs[sub_Phi]                                              
            quad_base[sub_Phi] = ((diffs_sub @ unique_inv_Phi[Phi_index]) * diffs_sub).sum(axis=1)  
        quad_form_values = nu_values * quad_base                             
        
        term3 = -0.5 * quad_form_values
        term4 = -0.5 * q / beta_values
        
        ElnNx_all_i = term1 + term2 + term3 + term4
        ELBOElnNx_i += np.sum(r_ik[:, k] * ElnNx_all_i)

        ElnNx_all_ik[:, k] = ElnNx_all_i 
    
    
    ELBOElnpc = 0
    for j in range(0, p):                                                           
        _G = Elnpsi_kjg[:, j, categoricaldataframe[modelled_categorical_columns[j]]]  
        ELBOElnpc += np.sum(r_ik * _G.T)                                             


    ELBOElnpz = 0
    for k in range(0, K):
        term_for_k = np.sum(r_ik[:, k] * Elnpi_k[k, pi_indices])
        ELBOElnpz += term_for_k 
                           

    ELBOElnqz = np.sum(xlogy(r_ik, r_ik))
    
    KLmugivenLambda=0
    for k in range(0,K):
        for g in allcatvalsmu[0]:
            KLmugivenLambdaincrement=-q*np.log(beta_kg_prior)+q*np.log(beta_kg_hat[iteration,k,catvaluetoindexmu[g]])-q+quadform(m_kg_hat[iteration,k,catvaluetoindexmu[g],:]-mu_kg_prior[k,catvaluetoindexmu[g],:],beta_kg_prior*nu_k_hat[iteration,k,catvaluetoindexmu[g]]*inv_Phi_k_hat_stored[k,catvaluetoindexSigma[g],:])+q*beta_kg_prior/beta_kg_hat[iteration,k,catvaluetoindexSigma[g]]
            KLmugivenLambda=KLmugivenLambda+0.5*KLmugivenLambdaincrement
        
  
    KLLambdadigterm=ElndetLambda_k_digammaterm

    KLLambda=0
    for k in range(0,K):
        for g in allcatvalsSigma[0]:
            KLLambdaincrement=nu*log_inv_det_Phi_prior-nu*log_inv_det_Phi_k_hat[k,catvaluetoindexSigma[g]]+nu_k_hat[iteration,k,catvaluetoindexSigma[g]]*np.trace(Phi @ inv_Phi_k_hat_stored[k,catvaluetoindexSigma[g],:] )+2*multigammaln(nu/2, q)-2*multigammaln(nu_k_hat[iteration,k,catvaluetoindexSigma[g]]/2, q)+(nu_k_hat[iteration,k,catvaluetoindexSigma[g]]-nu)*KLLambdadigterm[k,catvaluetoindexSigma[g]]-nu_k_hat[iteration,k,catvaluetoindexSigma[g]]*q
            KLLambda=KLLambda+0.5*KLLambdaincrement
    
    KLpi=0
    KLpiKsum=np.zeros(len(allcatvalspi[0])) 
    for k in range(0,K):
        for g in allcatvalspi[0]:
            KLpiKsum[catvaluetoindexpi[g]]=KLpiKsum[catvaluetoindexpi[g]]+gammaln(piparamprior)-gammaln(piparamprior+sum_r_pi_gk[g][k]) #OPTIMIZATION (4)+(5): cached locations/sums


    KLpiNKsum=np.zeros(len(allcatvalspi[0]))      
    for k in range(0,K):
        for g in allcatvalspi[0]:
            KLpiNKsum[catvaluetoindexpi[g]]=KLpiNKsum[catvaluetoindexpi[g]]+sum_r_pi_gk[g][k]*Elnpi_k[k,catvaluetoindexpi[g]]
    
    for g in allcatvalspi[0]:
        KLpi=KLpi+gammaln(K*piparamprior+totsum_r_pi_g[g])-gammaln(K*piparamprior)+KLpiKsum[catvaluetoindexpi[g]]+KLpiNKsum[catvaluetoindexpi[g]] #OPTIMIZATION (4)+(5): cached locations/sums
    
    
    if Psiindependentofk==1:
        KLPsi=0
        for j in range(0,p):
                KLPsiterm1=gammaln(np.sum(Psiparam[iteration,0,j,0:categorical_columns_dims[j]]))-gammaln(np.sum(Psiparamprior[0,j,0:categorical_columns_dims[j]]))
                KLPsiterm2=np.sum(gammaln(Psiparam[iteration,0,j,0:categorical_columns_dims[j]]))-np.sum(gammaln(Psiparamprior[0,j,0:categorical_columns_dims[j]]))
                KLPsiterm3=(Psiparam[iteration,0,j,0:categorical_columns_dims[j]]-Psiparamprior[0,j,0:categorical_columns_dims[j]])*( digamma(Psiparam[iteration,0,j,0:categorical_columns_dims[j]])-digamma(np.sum(Psiparam[iteration,0,j,0:categorical_columns_dims[j]])) )
                KLPsi=KLPsi+KLPsiterm1-KLPsiterm2+np.sum(KLPsiterm3) 
        
    else:
        KLPsi=0
        for k in range(0,K):
            for j in range(0,p):
                    KLPsiterm1=gammaln(np.sum(Psiparam[iteration,k,j,0:categorical_columns_dims[j]]))-gammaln(np.sum(Psiparamprior[k,j,0:categorical_columns_dims[j]]))
                    KLPsiterm2=np.sum(gammaln(Psiparam[iteration,k,j,0:categorical_columns_dims[j]]))-np.sum(gammaln(Psiparamprior[k,j,0:categorical_columns_dims[j]]))
                    KLPsiterm3=(Psiparam[iteration,k,j,0:categorical_columns_dims[j]]-Psiparamprior[k,j,0:categorical_columns_dims[j]])*( digamma(Psiparam[iteration,k,j,0:categorical_columns_dims[j]])-digamma(np.sum(Psiparam[iteration,k,j,0:categorical_columns_dims[j]])) )
                    KLPsi=KLPsi+KLPsiterm1-KLPsiterm2+np.sum(KLPsiterm3) 

    
    print('ELBO terms check')
    print(ELBOElnNx_i) 
    print(ELBOElnpc)
    print(ELBOElnpz)
    print(ELBOElnqz)
    print(-KLmugivenLambda)
    print(-KLLambda)
    print(-KLpi)
    print(-KLPsi)
    ELBO[iteration]=ELBOElnNx_i+ELBOElnpc+ELBOElnpz-ELBOElnqz-KLmugivenLambda-KLLambda-KLpi-KLPsi
    print('ELBO')
    print(ELBOElnNx_i+ELBOElnpc+ELBOElnpz-ELBOElnqz-KLmugivenLambda-KLLambda-KLpi-KLPsi)
    
    improvement=ELBOElnNx_i+ELBOElnpc+ELBOElnpz-ELBOElnqz-KLmugivenLambda-KLLambda-KLpi-KLPsi-ELBOold
    
    print('improvement')
    print(improvement)
    

    
    ELBOold=ELBOElnNx_i+ELBOElnpc+ELBOElnpz-ELBOElnqz-KLmugivenLambda-KLLambda-KLpi-KLPsi
    

    Elnpsisum_all_ik = np.zeros((N, K))
    Elnpi_all = np.zeros((N, K))
    for k in range(0, K):
        Elnpi_all[:, k] = Elnpi_k[k, pi_indices]

    if Psiindependentofk != 1:                                                    
        for j in range(0, p):                                                       
            Elnpsisum_all_ik += Elnpsi_kjg[:, j,                                  
                                           categoricaldataframe[modelled_categorical_columns[j]]].T   

    if Psiindependentofk == 1:
        logrho_ik = ElnNx_all_ik + Elnpi_all
    else:
        logrho_ik = ElnNx_all_ik + Elnpsisum_all_ik + Elnpi_all

    r_ik = softmax(logrho_ik, axis=1)
        
    
    outputr_ik = open('r_ik'+str(N)+'.pkl', 'wb') 
    pickle.dump(r_ik, outputr_ik)
    outputr_ik.close()

    print('saved r_ik up to iteration '+str(iteration))
    

    vi_compute_elapsed += time.perf_counter() - iter_t0
    timerecord[iteration] = vi_compute_elapsed 

    if iteration % 10 == 0:

        for g in allcatvals[0]:
            locations_g = np.where(combinedcategories == g)
            print('g is '+str(g))
            print('predicted hard clusters')
            print(Counter(np.argmax(r_ik[locations_g[0].tolist(),:], axis=1)))
            countordered=Counter(np.argmax(r_ik[locations_g[0].tolist(),:], axis=1)).most_common(K)
            clustersordered=[element for element, count in countordered]
    
    
    iteration=iteration+1
    dummyiteration=dummyiteration+1 
    
end_time = time.time()
elapsed_time = end_time - start_time

print('elapsed_time')
print(elapsed_time)


iteration=iteration-1

In [ ]:
#save posterior samples

import pickle

#output = open('data.pkl', 'wb')
#pickle.dump(my_data, output)
#output.close()

outputpiparam = open('piparam.pkl', 'wb')
pickle.dump(piparam, outputpiparam)
outputpiparam.close()

outputbeta_kg_hat = open('beta_kg_hat.pkl', 'wb')
pickle.dump(beta_kg_hat, outputbeta_kg_hat)
outputbeta_kg_hat.close()

outputm_kg_hat = open('m_kg_hat.pkl', 'wb')
pickle.dump(m_kg_hat, outputm_kg_hat)
outputm_kg_hat.close()

outputnu_k_hat = open('nu_k_hat.pkl', 'wb')
pickle.dump(nu_k_hat, outputnu_k_hat)
outputnu_k_hat.close()

outputPhi_k_hat = open('Phi_k_hat.pkl', 'wb')
pickle.dump(Phi_k_hat, outputPhi_k_hat)
outputPhi_k_hat.close()

outputPsiparam = open('Psiparam.pkl', 'wb')
pickle.dump(Psiparam, outputPsiparam)
outputPsiparam.close()

outputr_ik = open('r_ik.pkl', 'wb')
pickle.dump(r_ik, outputr_ik)
outputr_ik.close()

outputELBO = open('ELBO.pkl', 'wb')
pickle.dump(ELBO, outputELBO)
outputELBO.close()

print('done')

In [ ]:
import pickle

piparam_pkl = open('piparam.pkl', 'rb')
piparam = pickle.load(piparam_pkl)

beta_kg_hat_pkl = open('beta_kg_hat.pkl', 'rb')
beta_kg_hat = pickle.load(beta_kg_hat_pkl)

m_kg_hat_pkl = open('m_kg_hat.pkl', 'rb')
m_kg_hat = pickle.load(m_kg_hat_pkl)

nu_k_hat_pkl = open('nu_k_hat.pkl', 'rb')
nu_k_hat = pickle.load(nu_k_hat_pkl)

Phi_k_hat_pkl = open('Phi_k_hat.pkl', 'rb')
Phi_k_hat = pickle.load(Phi_k_hat_pkl)

Psiparam_pkl = open('Psiparam.pkl', 'rb')
Psiparam = pickle.load(Psiparam_pkl)

r_ik_pkl = open('r_ik.pkl', 'rb')
r_ik = pickle.load(r_ik_pkl)

ELBO_pkl = open('ELBO.pkl', 'rb')
ELBO = pickle.load(ELBO_pkl)

print('done')

 # Find the last index where ELBO is not equal to 1.0
index = np.max(np.where(ELBO != 1.0))
print(index)

iteration=index


In [ ]:
#radar plots for continuous variables

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t

def empirical_cdf(x_new, data):
    return np.mean(data <= x_new)

from math import floor, log10
def round_to_sigfigs(x, sigfigs=3):
    if x == 0:
        return 0
    else:
        return round(x, -int(floor(log10(abs(x)))) + (sigfigs - 1))

variable_names = modelled_continuous_columns

means = m_kg_hat[iteration,:,0,:]

variances=np.zeros((K,q))

for k in range(0,K):
    for col in range(0,q):
        
        shapemattop=Phi_k_hat[iteration,k,0,:]*(beta_kg_hat[iteration,k,0]+1) #subgroup removed
        shapematbottom=beta_kg_hat[iteration,k,0]*(nu_k_hat[iteration,k,0]-q+1)
        
        shape=shapemattop/shapematbottom
        
        variances[k,col]=shape[col,col]
        
upper_ci=0.9 #0.975
lower_ci=0.1 #0.025


quantiles_2_5 = means - t.ppf(upper_ci, nu_k_hat[iteration,:,0]-q+1)[:, np.newaxis] * np.sqrt(variances)  # Lower quantile (2.5%)
quantiles_97_5 = means + t.ppf(upper_ci, nu_k_hat[iteration,:,0]-q+1)[:, np.newaxis] * np.sqrt(variances)  # Upper quantile (97.5%)


meansprob=np.zeros((K,q))
quantiles_2_5_prob=np.zeros((K,q))
quantiles_97_5_prob=np.zeros((K,q))

for k in range(0,K):
    for col in range(0,q):
        if modelled_continuous_columns[col] in {"height", "eGFR", "hdl"}:
            meansprob[k,col]=empirical_cdf(-means[k,col], -fulldataframe[modelled_continuous_columns[col]])
            quantiles_2_5_prob[k,col]=empirical_cdf(-quantiles_97_5[k,col], -fulldataframe[modelled_continuous_columns[col]])
            quantiles_97_5_prob[k,col]=empirical_cdf(-quantiles_2_5[k,col], -fulldataframe[modelled_continuous_columns[col]])

        else:
            meansprob[k,col]=empirical_cdf(means[k,col], fulldataframe[modelled_continuous_columns[col]])
            quantiles_2_5_prob[k,col]=empirical_cdf(quantiles_2_5[k,col], fulldataframe[modelled_continuous_columns[col]])
            quantiles_97_5_prob[k,col]=empirical_cdf(quantiles_97_5[k,col], fulldataframe[modelled_continuous_columns[col]])

theta = np.linspace(0, 2 * np.pi, q, endpoint=False).tolist()
theta += theta[:1]  

fig, axs = plt.subplots(2, 5, figsize=(18, 8), subplot_kw=dict(polar=True),constrained_layout=True)
axs = axs.flatten()

for i in range(K):
    values_mean = np.append(meansprob[i], meansprob[i][0]) 
    values_2_5 = np.append(quantiles_2_5_prob[i], quantiles_2_5_prob[i][0])
    values_97_5 = np.append(quantiles_97_5_prob[i], quantiles_97_5_prob[i][0])
    
    axs[i].plot(theta, values_mean, label="Mean", color="blue")
    axs[i].fill_between(theta, values_2_5, values_97_5, color="green", alpha=0.25, label=str(100*(lower_ci))+" to "+str(100*(upper_ci))+"% Quantile")
    
    pi_kposteriormean=piparam[iteration,i,catvaluetoindexpi[g]]/np.sum(piparam[iteration,:,catvaluetoindexpi[g]]) 
    axs[i].set_title(f"Cluster {i+1} ," +str(round_to_sigfigs(pi_kposteriormean, 3)), fontsize=15)
    axs[i].set_xticks(theta[:-1])
    axs[i].set_xticklabels(variable_names)
    
    n_circles = 4
    axs[i].set_yticks(np.linspace(0, 1, n_circles + 1)) 
    axs[i].set_yticklabels([f"{int(i*100)}%" for i in np.linspace(0, 1, n_circles + 1)])
    
    
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)
plt.suptitle("Radar Plots for marginal posterior predictive means and "+str(100*(upper_ci-lower_ci))+"% credible interval in empirical distribution probability, per cluster", fontsize=18)
plt.tight_layout(rect=[0, 0, 1, 0.95],h_pad=0.1)
fig.subplots_adjust(hspace=0.1)

plt.savefig('radarplotsmiddleagedmen'+str(K)+'clusters'+str(upper_ci-lower_ci)+'.pdf')

plt.show()


In [ ]:
#bar plots for categorical variable

smokecounts=Counter(healthdata['smokecombined'])
smokelabels = [0, 1, 2]
smoketotal = sum(smokecounts.values())
smokeproportions = [smokecounts[label] / smoketotal for label in smokelabels]
print(smokeproportions)

rows = 2
cols = 5

colors = ["green", "blue", "orange"]
labels = ["never smoked", "former smoker", "current smoker"]

import matplotlib.lines as mlines
red_line = mlines.Line2D([], [], color='red', linestyle='dashed', linewidth=2, label='Empirical proportion')


fig, axes = plt.subplots(rows, cols, figsize=(12, 6), sharey=True)

axes = axes.flatten()

for j in range(0,p):
    for k in range(K):

        barplotmat=Psiparam[iteration,k,j,0:categorical_columns_dims[j]]/np.sum(Psiparam[iteration,k,j,0:categorical_columns_dims[j]])

        ax = axes[k]
        ax.bar(range(categorical_columns_dims[j]), barplotmat, color=colors, edgecolor='black')
        
        for i, prop in enumerate(smokeproportions):
            ax.hlines(y=prop, xmin=i-0.4, xmax=i+0.4, colors='red', linestyles='dashed', linewidth=2)
        
        ax.set_title(f"k={k+1}", fontsize=10)
        ax.set_ylim(0, 1)  
        ax.set_xticks(range(categorical_columns_dims[j]))
        ax.set_ylabel("Probability")

        for i, prob in enumerate(barplotmat):
            ax.text(i, prob+0.02, f"{prob:.3f}", ha='center', va='bottom', fontsize=8) 

    for ax in axes[K:]:
        ax.axis("off")
    
    proxy_bars = [plt.Rectangle((0, 0), 1, 1, color=color, edgecolor="black") for color in colors]
    fig.legend(proxy_bars + [red_line], labels + ['Empirical proportion'], loc='lower center', ncol=categorical_columns_dims[j]+1)


    plt.tight_layout()
    plt.savefig('barplotsmiddleagedmen'+str(K)+'clusters.pdf')
    plt.show()